# Resolução 02 — Mapas e território

**Curso:** Tecnologia, Dados e IA na Gestão Ambiental
**Encontro:** 5

Este notebook resolvedor cria um **mapa interativo** com Folium, adiciona pontos de coleta,
uma área verde e uma camada didática de focos de calor. Ele roda com **Runtime → Run all**;
os dados são baixados automaticamente.

> As coordenadas dos focos são **didáticas**: o CSV de exemplo é agregado por mês e bioma e
> não tem latitude/longitude. Em um projeto real, use a camada oficial do INPE/TerraBrasilis
> ou do MapBiomas.

## Passo 0 — Instalar o Folium e baixar os dados

O Colab normalmente já tem o Folium, mas a célula instala se necessário. O mesmo código
funciona localmente.

In [ ]:
from pathlib import Path
import urllib.request

try:
    import folium
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "folium"])
    import folium

BASE_DADOS = "https://raw.githubusercontent.com/Marquezbertin/Curso-Tecnologia-Dados-Gestao-Ambiental/main/06-DADOS/exemplos"

def baixar(nome, base=BASE_DADOS):
    destino = Path(nome)
    if not destino.exists():
        print(f"Baixando {nome}...")
        urllib.request.urlretrieve(f"{base}/{nome}", destino)
    print(f"Arquivo pronto: {destino.resolve()}")
    return destino

ARQUIVO_FOCOS = baixar("exemplo-focos-calor-sintetico.csv")
print("Folium:", folium.__version__)

## Passo 1 — Criar o mapa base

Substitua `lat` e `lon` pelas coordenadas do seu município. No Google Maps, clique com o
botão direito no ponto desejado e copie os números.

In [ ]:
lat, lon = -22.55, -47.90  # coordenada didática (interior de São Paulo)

mapa = folium.Map(
    location=[lat, lon],
    zoom_start=13,
    tiles="OpenStreetMap",
)
mapa

## Passo 2 — Adicionar os pontos do projeto

Cada ponto precisa de nome, coordenada e tipo. O tipo define a cor e a legenda.

In [ ]:
pontos = [
    {"nome": "Escola", "lat": -22.550, "lon": -47.900, "tipo": "coleta"},
    {"nome": "Ponto de coleta 1", "lat": -22.555, "lon": -47.905, "tipo": "coleta"},
    {"nome": "Ponto de coleta 2", "lat": -22.545, "lon": -47.892, "tipo": "coleta"},
    {"nome": "Área verde", "lat": -22.545, "lon": -47.895, "tipo": "verde"},
]

for p in pontos:
    folium.Marker(
        location=[p["lat"], p["lon"]],
        popup=f"{p['nome']} ({p['tipo']})",
        tooltip=p["nome"],
        icon=folium.Icon(
            color="green" if p["tipo"] == "verde" else "blue",
            icon="leaf" if p["tipo"] == "verde" else "ok-sign",
        ),
    ).add_to(mapa)

mapa

## Passo 3 — Camada de círculos

O raio está em metros. Círculos ajudam a mostrar área de influência, mas não substituem uma
análise de buffer com distância calculada.

In [ ]:
for p in pontos:
    folium.Circle(
        location=[p["lat"], p["lon"]],
        radius=300,
        color="#16724a" if p["tipo"] == "verde" else "#2b6cb0",
        fill=True,
        fill_opacity=0.15,
        popup=f"Raio didático de 300 m — {p['nome']}",
    ).add_to(mapa)

mapa

## Passo 4 — Desafio: camada de focos de calor

Vamos ler o CSV de focos, agregar por bioma e colocar uma camada no mapa. As posições são
**didáticas** e servem para praticar a leitura de camadas; o CSV não tem coordenadas reais.

In [ ]:
import pandas as pd

focos = pd.read_csv(ARQUIVO_FOCOS)
total_bioma = focos.groupby("bioma")["focos"].sum().sort_values(ascending=False)

print("Focos por bioma (2019–2024):")
print(total_bioma.to_string())

coordenadas_didaticas = {
    "Cerrado": (-22.558, -47.910),
    "Mata Atlantica": (-22.542, -47.888),
}

camada_focos = folium.FeatureGroup(name="Focos de calor (posições didáticas)")

for bioma, total in total_bioma.items():
    if bioma not in coordenadas_didaticas:
        continue
    folium.Circle(
        location=coordenadas_didaticas[bioma],
        radius=200 + total / 8,
        color="#b7791f",
        fill=True,
        fill_color="#f0a33a",
        fill_opacity=0.35,
        popup=f"{bioma}: {int(total)} focos (2019–2024)",
        tooltip=f"{bioma} — {int(total)} focos",
    ).add_to(camada_focos)

camada_focos.add_to(mapa)
mapa

## Passo 5 — Legenda, controle de camadas e exportação

Um mapa sem legenda não é evidência. A legenda abaixo é fixa no canto e o controle permite
ligar e desligar camadas.

In [ ]:
legenda = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 12px 14px; border: 1px solid #dde6de;
            border-radius: 8px; font-size: 12px; line-height: 1.6;
            box-shadow: 0 4px 14px rgba(0,0,0,0.15);">
  <b>Legenda</b><br>
  <span style="color:#2b6cb0;">●</span> Ponto de coleta<br>
  <span style="color:#16724a;">●</span> Área verde<br>
  <span style="color:#b7791f;">●</span> Focos de calor (didáticos)
</div>
'''

mapa.get_root().html.add_child(folium.Element(legenda))
folium.LayerControl(collapsed=False).add_to(mapa)
mapa.save("mapa-com-focos.html")
print("Mapa salvo em: mapa-com-focos.html")
mapa

## Passo 6 — Interpretação com respostas

### O que os dados mostram

- **Cerrado:** 2.926 focos no período 2019–2024.
- **Mata Atlântica:** 1.617 focos no mesmo período.
- **Setembro** é o mês de pico (726 focos), seguido de agosto (630) e outubro (615).
- O ano com mais focos é **2022** (831); o com menos é **2023** (700).
- O maior registro mensal é **99 focos** no Cerrado em setembro de 2022.

### O que o mapa mostra e o que não mostra

- Mostra a **posição didática** dos pontos e a sobreposição visual com a área verde.
- **Não mostra** a localização real dos focos: o CSV é agregado e não tem lat/lon.
- **Não mostra** área queimada, causa, velocidade do vento nem direção do fogo.
- **Não prova** que os pontos de coleta estão dentro da área verde. Para isso, é preciso
  calcular distância/buffer com coordenadas reais.

### Comparação com os pontos de coleta

A célula abaixo calcula a distância entre a escola e os demais pontos. Todos ficam a menos de
1 km, mas isso é consequência das coordenadas didáticas. Em um projeto real, use as
coordenadas medidas com GPS ou extraídas do mapa e registre a precisão.

In [ ]:
from math import radians, sin, cos, asin, sqrt

def distancia_km(lat1, lon1, lat2, lon2):
    raio_terra = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * raio_terra * asin(sqrt(a))

escola = pontos[0]
for p in pontos[1:]:
    d = distancia_km(escola["lat"], escola["lon"], p["lat"], p["lon"])
    print(f"{p['nome']}: {d:.2f} km da escola")

## Passo 7 — Como usar dados reais de focos

Para substituir a camada didática por dados reais:

1. baixe a camada de focos do INPE/TerraBrasilis ou use o MapBiomas;
2. confira o sistema de coordenadas (graus decimais, WGS84);
3. carregue com `geopandas` ou `folium.GeoJson`;
4. registre a data de acesso, o satélite e a resolução;
5. compare com um buffer ao redor da escola e escreva a limitação.

Exemplo de esqueleto (não roda sem o arquivo real):

```python
# import geopandas as gpd
# focos_reais = gpd.read_file("focos_2024.shp")
# folium.GeoJson(focos_reais).add_to(mapa)
```

## Ficha de fonte do mapa

| Campo | Resposta |
|---|---|
| Base do mapa | OpenStreetMap |
| Pontos | Coordenadas didáticas do exemplo |
| Camada de focos | CSV sintético agregado por bioma/mês, sem coordenadas reais |
| Período | 2019–2024 |
| Limitação | Posições didáticas; não representa focos reais no município |
| Uso recomendado | Praticar camadas, legenda e leitura espacial |
| Uso não recomendado | Laudo, alerta ou decisão de fiscalização |

## Checklist de entrega

- [ ] O mapa abre e permite ligar/desligar camadas.
- [ ] Cada ponto tem nome e tipo.
- [ ] A legenda explica cor, ponto e camada.
- [ ] A fonte das coordenadas está escrita.
- [ ] O período dos dados está escrito.
- [ ] Há uma limitação explícita sobre o mapa.

**Próximo notebook resolvido:** [`03-resolucao-ia-classificacao-ambiental.ipynb`](03-resolucao-ia-classificacao-ambiental.ipynb)